# ICS 604: APPLIED DATA SCIENCE

## Introduction to Time Series Analysis

- ### Time series data
- ### Pandas and time series
- ### Modeling the trend and seasonality
---

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Introduction to Time Series Analysis

Time series analysis focuses on understanding and modeling data points collected over time, where the temporal ordering carries important information. In this introduction, we will cover a few essential concepts that form the foundation for analyzing and modeling time series data. These concepts will help you recognize patterns, understand underlying structures, and begin building simple forecasting models.

It is important to note that this is only an introductory overview. Time series analysis is a rich and complex field that spans a wide range of methods, from classical statistical techniques to modern machine learning approaches, and typically warrants an entire course of study. Here, our goal is to build intuition and familiarize you with key ideas rather than explore the topic exhaustively.

For simplicity, we will focus exclusively on univariate time series, where a single variable is observed over time. This allows us to concentrate on core concepts such as trends, seasonality, and noise without the added complexity of interactions between multiple variables.

### What Are Time Series?

Time series are datasets in which the values of a variable are recorded sequentially over time, allowing us to observe how that quantity evolves. Unlike static datasets, the temporal dimension is central, as it captures patterns such as growth, cycles, or fluctuations. Common examples of time series include the number of cars passing through an intersection measured at regular intervals, the number of travelers arriving at HNL over time, or temperature readings recorded throughout the day or across seasons.

#### Time Series Data

Time series data differs from the types of data we have considered so far in two fundamental ways. First, observations are typically not independent of one another. The value observed at time $t$ is often influenced by the value at time $t−1$, meaning that past behavior can inform present and future values. As a result, the ordering of observations is crucial and cannot be ignored or shuffled without losing important information.

Second, time series data is not necessarily identically distributed. The statistical properties of the data — such as the mean or variance — may change over time. While the data may still belong to the same general class of distributions, the parameters governing those distributions can shift, reflecting underlying changes in the process being observed. These characteristics make time series analysis both more challenging and more informative than standard data analysis.

### Some Goals of Time Series Analysis or Modeling

One of the primary goals of time series analysis is to develop a model that adequately describes the observed data and captures its underlying structure. With such a model, we can identify important patterns such as trends or recurring behaviors. These models are also useful for forecasting future values and for quantifying the uncertainty or variability associated with those forecasts.

In this introduction, we will focus on two broad classes of time series models. The first consists of regression-based models, where time (or functions of time) is used as an explanatory variable to help predict the response variable $y$. These models are useful for capturing systematic patterns like trends or seasonality.

The second class includes models that rely on past values of the series and past prediction errors to model current observations. These approaches form the backbone of many applications in fields such as econometrics, finance, and atmospheric science, where understanding temporal dependence is essential for accurate modeling and forecasting.

### Types of Time Series

Time series can vary widely in structure, but they often fall into two broad categories.

**1. Random Walks:**

Random walks are dominated by noise, with no systematic patterns over time. Any apparent trends or structures are generally the result of random fluctuations rather than an underlying process. An example of this type is daily stock returns. Mathematically, a random walk can be expressed as:
   
$$
v_t= v_{t-1} + \epsilon_t, \hspace{0.2in} \epsilon_t ~\sim \mathcal{N}(\mu, \sigma)
$$ 

Here, each value depends only on the previous value plus some random noise $\epsilon_t$, which is typically drawn from a normal distribution. Because of this, predicting future values beyond very short horizons is challenging, as the series is largely unpredictable.

**2. Combination of Trend, Seasonality, and Noise:**

Many real-world time series are structured as combinations of a systematic trend, repeating seasonal patterns, and random noise. For instance, atmospheric $\mbox{CO}_2$ levels exhibit both an upward trend and annual seasonal cycles. This type of series can be modeled as:

$$
v_t = f(t) + s(t) + \epsilon_t
$$

where $f(t)$ represents the trend component, $s(t)$ captures seasonality, and $\epsilon_t$ is random noise.


In [ ]:
### Types of Time series: A Random Walk

v = np.zeros(1000)
mu = 0
sigma = 1

for t in range(1000):
    v[t] = v[t-1] + np.random.normal(mu, sigma)

v[0:10]

In [ ]:
plt.figure(figsize=(18, 6))

plt.plot(np.arange(1000), v)
plt.title("Example of a Ramdom Walk", fontsize=20);

In [ ]:
### Types of Time series: Combination of trend, seasonality, and noise

x_axis = np.arange(0, 20, 0.25)

trend = np.zeros(len(x_axis))
seasonaility = np.zeros(len(x_axis))
noise = np.zeros(len(x_axis))

a = 2 
b = 0.3

trend = a + b * x_axis
seasonaility = np.sin(x_axis)
noise = np.random.normal(0, 0.5, len(x_axis))    

plt.figure(figsize=(20, 6))

plt.subplot(1, 3, 1)
plt.plot(x_axis, trend)
plt.title("trend", fontsize=18)

plt.subplot(1, 3, 2)
plt.plot(x_axis, seasonaility)
plt.title("seasonality", fontsize=18)

plt.subplot(1, 3, 3)
plt.plot(x_axis, noise)
plt.title("noise", fontsize=18);

In [ ]:
### Types of Time series: Combination of trend, seasonality, and noise

plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
plt.plot(x_axis, trend, alpha=0.7)
plt.subplot(1, 3, 2)
plt.plot(x_axis, trend + seasonaility, c='r', alpha=0.7)
plt.subplot(1, 3,3)
plt.plot(x_axis, trend + seasonaility + noise, linewidth=2, c='k');

In [ ]:
### Types of Time series: Combination of trend, seasonality, and noise

plt.plot(x_axis, trend, alpha=0.7)
plt.plot(x_axis, trend + seasonaility, c='r', alpha=0.7)
plt.plot(x_axis, trend + seasonaility + noise, linewidth=2, c='k');

### Time Series Example: Mauna Loa Dataset

A classic example of a time series with trend, seasonality, and noise is the atmospheric $\mbox{CO}_2$ measurements recorded at Mauna Loa on the Big Island of Hawai'i. This dataset consists of monthly observations spanning from 1959 to 1990. It provides a good illustration of how real-world time series are often composed of multiple underlying components that evolve over time.

$$ 
\text{Time} ~~ \text{Series} ~~\overline{\underline{\text{is composed of}}} ~~Trend + Seasonality + Stochastic ~ Fluctuations
$$

Here:

- **Trend** captures the long-term increase in atmospheric $\mbox{CO}_2$ levels.
- **Seasonality** reflects recurring yearly patterns caused by seasonal carbon uptake and release by vegetation.
- **Stochastic Fluctuations** represent random noise or irregular variations that cannot be attributed to trend or seasonality.

In [ ]:
co2_data = pd.read_csv("data/carbon_dioxide.txt", names=["co2_val"])
co2_data

In [ ]:
co2_data.co2_val

## `pandas` and Time Series 

The `pandas` library provides powerful and convenient tools for working with time series data. It includes extensive functionality for tasks such as aligning multiple time series, resampling data at different frequencies, and handling missing values through imputation. While we will cover some of the most commonly used features, a more comprehensive treatment can be found in [*Python for Data Analysis*](https://wesmckinney.com/book/) by Wes McKinney, which includes an in-depth chapter on time series analysis. You may also refer to the pandas documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html.

To work effectively with time series in `pandas`, we often assign a `datetime` index to our data. For example, we can create a monthly date range starting in January 1959 and set it as the index of our dataset:

In [ ]:
# Generates ranges as timestamps, which are more appropriate for time series data
# Index is of type `pandas Timestamps`

dateIndex = pd.date_range('1/1/1959', periods=len(co2_data), freq="MS")
dateIndex

In [ ]:
print(co2_data.index)

In [ ]:
# datetime64
co2_data.index = dateIndex
print(co2_data.index)

In [ ]:
co2_data.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(co2_data['co2_val']);

### Indexing `datetime`

Once the index is converted to a `datetime64` type, pandas enables intuitive and flexible indexing. One of the most useful features is the ability to access data using partial date strings. You can select observations by year, month, or specific date ranges without needing exact matches:

In [ ]:
co2_data.loc['1960'] # Matches all dates in 1960, you don’t need to type each date exactly

In [ ]:
co2_data.loc['1960-4'] # Matches all dates in April 1960, even if the day isn’t specified

In [ ]:
co2_data.loc['1960-9':'1961-3'] # Matches all dates from September 1960 to March 1961

<br>

Notably, when slicing with date ranges, the end date is included in the result. This makes it easy to subset and visualize specific time periods:

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(co2_data.loc['1960':'1962'])
plt.grid()

### Accessing Subsets of the DateTime Object

Since the index of a time series in pandas is a timestamp (i.e., a specific point in time), we can easily extract and work with different components of that timestamp. This makes it straightforward to analyze patterns at different temporal resolutions or to filter data based on calendar attributes.

For example, we can directly access elements such as the day, day of the week, month, or even the name of the month or weekday. These attributes are especially useful for grouping data, identifying seasonal effects, or creating new features for modeling.

In [ ]:
co2_data.index.day

In [ ]:
co2_data.index.dayofweek

<br>

We can convert timestamps into more readable string formats using `strftime`:

In [ ]:
# Format code in the datetime docs: 
# https://docs.python.org/3/library/datetime.html#format-codes
# Formatting playground on https://www.strfti.me/

co2_data.index.strftime('%A')   # full weekday name

In [ ]:
co2_data.index.strftime('%b')   # abbreviated month name

In [ ]:
co2_data.index[:3].strftime('%a, %B %d, %Y')

<br>

We can filter data based on specific years or combinations of years:

In [ ]:
co2_data[co2_data.index.year.isin([1960, 1970])]

In [ ]:
sample_4years = co2_data[co2_data.index.year.isin([1960, 1970, 1980, 1990])]
display(sample_4years.head())
display(sample_4years.tail())

In [ ]:
print(sample_4years.index.year)
print("----------")
print(len(sample_4years.index.year))

In [ ]:
sample_4years.loc['1960']

These capabilities allow us to efficiently subset or transform time series data based on meaningful calendar components, which is particularly helpful when exploring trends, seasonality, or cyclical behavior.

### Modeling the Trend and Seasonality

Modeling time series data typically begins with understanding its underlying structure. This process can be broken into three main steps: exploring the data, forming a hypothesis about its components, and then specifying and testing a model that captures the observed trend and seasonality. While we aim to explain systematic patterns, it is important to recognize that the residual noise component is inherently random and cannot be modeled deterministically.

### 1. Exploring the Data

Exploratory data analysis is a critical first step, especially for time series. At this stage, the goal is to visually and statistically examine the data to uncover patterns and guide model development. Some key questions to consider include:

- Does the data exhibit seasonality?
- Where do the major peaks and troughs occur?
- Is the seasonal pattern stable across different time periods?
  - How much variability exists over time?

This stage is inherently open-ended — any technique that helps reveal structure in the data is useful.

In [ ]:
plt.figure(figsize=(15, 5))

years = [1960, 1970, 1980, 1990]

for i, year in enumerate(years):
    plt.subplot(2, 2, i+1)
    plt.plot(sample_4years.loc[str(year)]['co2_val'])

<br>
The observed pattern is roughly sinusoidal with a one-year period, showing that values rise and fall in a regular annual cycle. This initial analysis covers only four years of data, suggesting a consistent seasonal signal. Let’s now examine the entire dataset.

In [ ]:
# Examine seasonal trend in the data

co2_data['Month'] = co2_data.index.strftime('%b')
co2_data['Year'] = co2_data.index.year

co2_data.head()

In [ ]:
co2_data_piv = co2_data.pivot(index='Year', columns='Month', values='co2_val')

co2_data_piv.head()

In [ ]:
# We reindex the data to have the months in order

orderedMonths = ["Jan", "Feb", "Mar", 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

co2_data_piv = co2_data_piv.reindex(columns=orderedMonths)
co2_data_piv.head()

In [ ]:
co2_data_piv.shape

In [ ]:
plt.figure(figsize=(8, 4))

sns.boxplot(data=co2_data_piv)
plt.title('Distributions of $CO_2$ Values for Each Month from 1959 to 1999')
plt.ylabel('$CO_2$ Values');

### 2. Forming a Hypothesis

After exploring the data visually, the next step is to form a hypothesis about its underlying structure. Based on the earlier analysis, there is strong evidence to suggest that the time series exhibits a clear seasonal pattern. This hypothesis is supported by multiple visualization techniques, including both line plots and boxplots.

Even though only four years of data were examined in detail, both representations consistently indicate the presence of a repeating seasonal signal. The agreement between these two independent methods strengthens our confidence that the observed seasonality is not due to random variation, but rather reflects a systematic and recurring pattern in the data.

From this, we can hypothesize that the time series contains a stable seasonal component — likely with a yearly cycle — alongside a broader trend and random noise. This hypothesis will guide the next step, where we formally specify and test a model that incorporates these components.

<center><img src="https://www.dropbox.com/scl/fi/lkzuyifxtmtxr6kg2eczs/seasonality.png?rlkey=j8o51etry81kojq0hd308gjh6&st=z3s3uoyg&dl=1" width="1000"/>
<center><i>The seasonality looks sinusoidal with a period of one year.</i></center>


### Exploring the Trend

As with seasonality, identifying the trend is an exploratory process. A natural starting point is to use visualizations that summarize how the data evolves over longer time periods. One simple and effective approach is to group observations by year and compare their distributions.

For example, we can construct a boxplot of $\mbox{CO}_2$ levels by year:

In [ ]:
sample_4years = pd.DataFrame({"co2_val": sample_4years["co2_val"], 
                              'year': sample_4years.index.year})
sample_4years.head()

In [ ]:
plt.figure(figsize=(8, 4))

sns.boxplot(x="year", y="co2_val", data=sample_4years);

This visualization allows us to compare the distribution of values across different years. If the median and overall distribution shift upward (or downward) over time, this provides evidence of a trend.

In this case, if the boxplots show increasing median $\mbox{CO}_2$ levels from earlier years (e.g., 1960) to later years (e.g., 1990), it suggests a clear upward trend in the data. Unlike line plots, which show point-by-point changes, boxplots summarize the distribution within each year, making long-term shifts easier to detect.

This type of visualization is especially useful because it separates short-term variability (captured within each box) from long-term movement (captured across boxes), helping us clearly identify the presence of a trend.

#### Averaging / Aggregating Data Per Period

Another effective way to explore the overall trend in a time series is to aggregate the data over a larger time period, such as yearly values. By summarizing the data in this way, we reduce short-term fluctuations and make long-term patterns more visible.

To do this in pandas, we group the data by year. Instead of using the standard `groupby` approach, time series data allows us to use the more specialized `resample` method. This method is designed specifically for datetime-indexed data and lets us specify how we want to aggregate observations over time.

In this example, we use `'YS'` (Year Start) to aggregate values annually:

In [ ]:
annual_sums = co2_data['co2_val'].resample('YS').sum()
annual_sums

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(np.arange(len(annual_sums)) + 1959, annual_sums)
plt.title('Aggregated Annual $CO_2$ Values from 1959 to 1999')
plt.ylabel('Sum of Annual $CO_2$ Values')
plt.xlabel('Year');

### 2. Forming a Hypothesis

Based on the exploratory visualizations, there is clear evidence of a trend in the data. Both the yearly boxplots and the aggregated annual values indicate a consistent pattern over time.

Specifically, the data shows a **continuous increase in $\mbox{CO}_2$ concentrations**, suggesting the presence of a strong upward trend. This pattern is visible regardless of the visualization method used, which increases our confidence that the trend is real and not an artifact of a particular plotting technique.

Using multiple visualizations — such as boxplots to compare yearly distributions and aggregated plots to observe overall movement — provides complementary perspectives on the same phenomenon. Together, they reinforce the hypothesis that the time series contains a systematic and increasing trend component, which should be incorporated into any model we build.

### 3. Specifying and Testing a Model

### Some Useful Time Series Modeling Tips

When building time series models, a few simple preprocessing steps can make modeling more intuitive and effective.

1. **Shift the data to pass through the origin**<br>
It is often helpful to transform the data so that it starts near zero. This makes it easier to define and interpret functions that describe the trend. Practically, this can be done by subtracting a constant offset (such as the first observation or the mean) from the entire series. This “centering” simplifies both visualization and model formulation, although this is not always necessary — whether or not you should shift depends on the contex.

<center><img src="https://www.dropbox.com/scl/fi/hhwr5b6cjag5x6p05gqkh/before_after-2.png?rlkey=xfzjx4qmfgfthbhab1hg6kjx8&st=rvijeggr&dl=1" width=500/></center>

2. **Convert time into a numerical variable**<br>
Since most models require numerical inputs, working directly with dates or string-based timestamps can be cumbersome. Instead, we can represent time as a sequence of numbers — for example, the number of months since the first observation. This creates a clean, continuous variable that can be easily used in regression or other modeling approaches.

Together, these steps help standardize the data and make it more suitable for mathematical modeling, especially when fitting trend or seasonal components.

In [ ]:
normalized_col = pd.DataFrame(co2_data['co2_val'] - co2_data['co2_val'].iloc[0])
normalized_col.index = range(0, len(normalized_col))

plt.figure(figsize=(12, 4))
plt.plot(normalized_col.index, co2_data['co2_val'], alpha=0.5, label="Before")
plt.plot(normalized_col['co2_val'], label="After")
plt.xlim(0, 400)
plt.legend(fontsize=14);

### Using an Appropriate Model for the Trend

Now that we have established the presence of a trend, the next step is to choose a functional form that can adequately describe it. A natural question to ask is: *what kind of function best captures the observed increase?*

From visual inspection, the trend does not appear to be strictly linear. Instead, the rate of increase seems to change over time, suggesting that a more flexible functional form may be appropriate.

One possible candidate is a **power law model**, given by:

$$
    f(x) = c_0 + c \cdot x^m 
$$

This type of function allows for nonlinear growth depending on the exponent $m$. By adjusting the parameters $c_0$, $c$, and $m$, we can capture a wide range of behaviors, from near-linear to more accelerated growth patterns.

More broadly, it is useful to be familiar with common **parent functions** (e.g., linear, polynomial, exponential, logarithmic, trigonometric, etc.). These serve as starting points when modeling trends. In practice, we often begin with a simple functional form and then iteratively refine it based on how well it fits the data.

The key idea is not to immediately find the perfect model, but to start with a reasonable approximation and improve it as we gain more insight into the structure of the time series.

In [ ]:
def func_powerlaw(x, m, c, c0):
    return c0 + c * x**m

x = np.arange(0, 400, 1)

y_1 = func_powerlaw(x, 1.5, 2, 0)
y_2 = func_powerlaw(x, 1.5, 3, 0)
y_3 = func_powerlaw(x, 1., 2, 0)

plt.figure(figsize=(5, 4))

plt.plot(x, y_1, label="$2 \\cdot x^{1.5}$")
plt.plot(x, y_2, label="$3 \\cdot x^{1.5}$")
plt.plot(x, y_3, label="$2 \\cdot x$")
plt.xlim(0, 400)
plt.legend();

### What is a Time Series Noise?

In the context of time series, noise refers to random, unexplained variation in the data that cannot be attributed to systematic components like trend or seasonality. It is typically modeled as an additive signal with the following properties:

- A constant mean (often assumed to be zero)
- A constant standard deviation
- Independence across time (i.e., noise at one time point does not depend on previous values)

This type of noise is often referred to as white noise and represents the purely random part of the series that remains after accounting for structured patterns.

### Working with Noisy Data

Noise is an inevitable part of most real-world time series, and handling it effectively is a key part of modeling. Interestingly, what is considered “noise” can depend on the goal of the analysis. For example, if we are primarily interested in modeling the long-term trend, then seasonal variation may itself be treated as a form of noise.

A simple approach to reducing noise — such as aggregating the data by year (e.g., averaging or summing) — can help smooth out fluctuations. However, this comes with a significant drawback: it reduces the number of data points. For instance, converting monthly data into yearly aggregates discards much of the original information, leaving fewer observations for model training and limiting our ability to capture finer patterns.

Because of this trade-off, more refined techniques are often preferred. These include methods that preserve the original data while separating components, such as detrending or deseasonalizing the series, allowing us to reduce noise without sacrificing valuable information.

### Smoothing with a Sliding Window

Smoothing is a technique used to highlight the essential patterns in time series data by reducing noise. The idea is to “correct” each data point by recomputing it using neighboring values within a specified window. One common approach is to take the mean of all points within the window, known as a sliding window (or moving average). Variants include weighted windows, where nearby points have a greater influence than those farther away.

When using a sliding window, each point is recalculated based on all observations within that window. This differs from simple aggregation (like yearly averages), where multiple observations are compressed into a single value. For example, with a window size of 12 (months), three years of monthly data would still produce many smoothed points (rather than just three), preserving more of the dataset’s structure.

<center><img src="https://www.dropbox.com/scl/fi/bqk79uwqov5xs1caz2ue7/smoothing.png?rlkey=6v74rvx1lk7vhf18wvbqzu221&st=lahu2uxw&dl=1"/>

In [ ]:
x_axis = normalized_col.index
y_axis = normalized_col['co2_val'].rolling(window=12, center=True).mean()

plt.figure(figsize=(12, 5))
plt.plot(x_axis, y_axis, 'r', label="window smoothed")
plt.plot(x_axis, normalized_col['co2_val'], alpha=0.5, label="original signal")
plt.title("Unweighted sliding-average smoothed data vs. original data", fontsize=10)
plt.legend();

<br>

#### What Happens When We Change the Window Size?

The size of the sliding window has a significant impact on the resulting smoothed series:

- **Larger window size:**
Using a larger window increases the level of smoothing. This reduces noise more aggressively and makes long-term trends easier to see. However, it can also **oversmooth** the data, potentially hiding important features such as seasonal patterns or sudden changes.

- **Smaller window size:**
A smaller window produces less smoothing. The resulting series stays closer to the original data, preserving short-term fluctuations and seasonal effects. However, more noise remains, which can make it harder to clearly identify underlying trends.

In practice, choosing the window size involves a trade-off between **noise reduction** and **detail preservation**. The appropriate choice depends on the specific goal — whether we are more interested in long-term trends or short-term variations.  

In [ ]:
y_axis_8 = normalized_col['co2_val'].rolling(window=8, center=True).mean()
y_axis_24 = normalized_col['co2_val'].rolling(window=24, center=True).mean()

plt.figure(figsize=(20, 5))

plt.subplot(1, 3, 1)
plt.plot(x_axis, y_axis, 'r', label="window=12")
plt.legend(fontsize=16)
plt.subplot(1, 3, 2)
plt.plot(x_axis, y_axis_8, 'r', label="window=8")
plt.legend(fontsize=16)
plt.subplot(1, 3, 3)
plt.plot(x_axis, y_axis_24, 'r', label="window=24")
plt.legend(fontsize=16);

In [ ]:
y_axis_17 = normalized_col['co2_val'].rolling(window=17, center=True).mean()

plt.figure(figsize=(16, 4))

plt.subplot(1, 2, 1)
plt.plot(x_axis, y_axis, 'r', label="window=12")
plt.legend(fontsize=16)
plt.subplot(1, 2, 2)
plt.plot(x_axis, y_axis_17, 'r', label="window=17")
plt.legend(fontsize=16);

<br>

When applying a sliding window, it is important to note that some observations at the boundaries are lost. For example, with a window of size 12, the first 6 and last 5 observations cannot be computed because there are not enough neighboring points to form a complete window. As a result, the smoothed series is slightly shorter than the original.

Once we obtain this smoother version of the data, we can more clearly observe and model the underlying trend. Earlier, we hypothesized that a power law function might provide a reasonable approximation. However, in practice, selecting the right functional form often requires experimentation. It is common to try multiple candidate models and compare how well they fit the data.

For instance, in addition to a power law, we might consider an exponential model of the form:

$$
f(x) = c_0 + c \cdot m^x
$$

The key question then becomes: *which model best represents the data?* One way to answer this is by evaluating how closely each model’s predictions match the observed values. This can be done using measures such as the mean squared error (MSE) or by visually comparing fitted curves to the smoothed data.

Ultimately, model selection is an iterative process — testing different functional forms, assessing their fit, and choosing the one that balances accuracy with simplicity.

In [ ]:
def func_exponential(x, m, c, c0):
    return c0 + c * (m**x)

x = np.arange(0, 400, 1)

y_1 = func_exponential(x, 1.01, 2, 0)

plt.figure(figsize=(5, 4))

plt.plot(x, y_1, label="$2 \\cdot {1.01}^x$")
plt.xlim(0, 400)
plt.legend();

In [ ]:
def func_exponential(x, m, c, c0):
    return c0 + c * (m**x)

x = np.arange(0, 400, 1)

y_1 = func_exponential(x, 1.01, 2, 0)
y_2 = func_exponential(x, 1.02, 2, 0)

plt.figure(figsize=(5, 4))

plt.plot(x, y_1, label="$2 \\cdot {1.01}^x$")
plt.plot(x, y_2, label="$2 \\cdot {1.02}^x$")
plt.xlim(0, 400)
plt.legend();

### Selecting the Appropriate Model

When choosing a model for the trend, it is not enough to consider only the general shape of the function — we must also consider how stable and interpretable the model is when fitting it to data.

Although an exponential function may appear to have a suitable shape for the observed trend, it can be difficult to work with in practice. In particular, its parameters (such as $m$) can be highly sensitive: small changes in their values may lead to large changes in the fitted curve. This sensitivity makes the model unstable and can result in poor fits, especially when working with real, noisy data.

In contrast, a power law model provides a more stable and flexible alternative:

$$
    f(x) = c_0 + c \cdot x^m 
$$

This form tends to be easier to fit and less sensitive to small parameter changes, making it more robust for capturing gradual, nonlinear trends. For this reason, the power law model is more appropriate for describing the trend in this dataset.

### Modeling the Smoothed Data

Once we have identified a suitable functional form — such as the power law — we can proceed to fit the model to the smoothed data. One approach would be to manually adjust the parameters $c_0$, $c$, and $m$ until the curve visually aligns well with the data. While this can build intuition, it is not efficient or precise, especially for more complex datasets.

In [ ]:
plt.plot(x_axis, func_powerlaw(x_axis, 1.8, 0.01, 0), label='Test 1 ($0.01 \\cdot x^{1.8}$)')
plt.plot(x_axis, func_powerlaw(x_axis, 0.9, 1, 0), label='Test 2 ($1 \\cdot x^{0.9}$)')
plt.plot(x_axis, func_powerlaw(x_axis, 1.5, 0.01, 0), label='Test 3 ($0.01 \\cdot x^{1.5}$)')
plt.plot(x_axis, y_axis, linewidth=6, alpha=0.6, label="True data")
plt.xlim(0, 400)
plt.legend();

<br>

A more systematic approach is to use numerical optimization methods. In Python, the `curve_fit` function from `scipy.optimize` provides a convenient way to estimate the best-fitting parameters. Documentation can be found here: https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

The `curve_fit()` function works by taking in the data along with a specified model (e.g., the power law function) and then optimizing the parameters to minimize the difference between the model and the observed data. It returns:

- `popt`: the optimal parameter values ($c_0$, $c$, $m$) that best fit the data
- `pcov`: the estimated covariance matrix of the parameters, which gives insight into the uncertainty of the estimates

In this context, we will primarily focus on `popt` and visually assess how well the fitted curve matches the smoothed data, rather than relying heavily on the covariance matrix.

It is also important to note that numerical optimization algorithms tend to perform better when parameter values are within a reasonable scale. Extremely large or very small parameter values can lead to instability, where even minor changes produce large deviations in the fitted curve. Proper scaling and reasonable initial guesses can significantly improve the quality of the fit.

In [ ]:
popt, _ = curve_fit(func_powerlaw, x_axis[6:-5], y_axis[6:-5], maxfev=2000)
popt

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(x_axis, y_axis, 'ro', alpha=0.1, label="original data")
plt.plot(x_axis, func_powerlaw(x_axis, *popt), c='k', linewidth=2, label='modeled data')
plt.legend();

### Subtracting the Trend

To model the seasonal component, it is common practice to first remove the trend from the data. This process, often called *detrending*, allows us to isolate the periodic behavior without interference from long-term growth. This is similar in spirit to what we did earlier when smoothing helped reduce seasonality to better observe the trend. Now, we reverse the perspective: since we have already modeled the trend, we subtract it from the original data at each time point.

In [ ]:
f, ax = plt.subplots(2, figsize=(12, 8))

ax[0].plot(x_axis, func_powerlaw(x_axis, *popt))
ax[1].plot(x_axis, normalized_col['co2_val'], label="before detrending")

detrended = normalized_col['co2_val'] - func_powerlaw(x_axis, *popt)
ax[1].plot(x_axis, detrended, label="after detrending")
plt.legend();

<br>

After removing the trend, the remaining signal (detrended) primarily reflects **seasonality and noise**. Importantly, the resulting signal exhibits a repeating, periodic structure. This makes it much easier to model the seasonal behavior independently, without needing to account for long-term increases.

### Modeling the Periodic Signal

With the trend removed, we can now focus on modeling the periodic structure of the data. A natural choice for capturing periodicity is a sine function:

$$
y = a \cdot sin(b\cdot x)
$$ 

- `a` controls the **amplitude**, or the height of the oscillations
- `b` controls the **frequency**, and therefore the period of the cycle

<center><img src="https://www.dropbox.com/scl/fi/b0w2g2q2g294tl92w3kpl/sin_cos.png?rlkey=xy3f6inuakxdpeab4uqhmb4dj&st=r2sa1wfz&dl=1" width="500px"></center>

In [ ]:
plt.axhline(y=0, linewidth=2, color='g', alpha=.3)
plt.axvline(x=0, linewidth=2, color='g', alpha=.3)
plt.xlim(-10, 10)
plt.plot(np.arange(-10, 10, 0.1), np.sin(np.arange(-10, 10, 0.1)), label="sine")
plt.plot(np.arange(-10, 10, 0.1), np.cos(np.arange(-10, 10, 0.1)), label="cosine")
plt.legend();

<br>

Since the natural period of the sine function is $2\pi$, we adjust it to match the data. In this case, the data is meteorological and exhibits a yearly cycle (i.e., every 12 months). To reflect this, we scale the input:

$$\text{sin}\left(\frac{2\pi x}{12}\right)$$

This ensures that the function repeats every 12 time steps (months), since:

$$\text{sin}\left(\frac{2\pi\cdot 12}{12}\right) = \text{sin}(2\pi) = 0$$

and similarly for $x=24,36,\dots$

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(x_axis, np.sin(2 * np.pi * x_axis / 12 ), 'r--', label='sine')
plt.plot(x_axis, detrended, label="data", alpha=0.5)
plt.legend();

<br>

From visual inspection, the amplitude appears to be approximately 3. Therefore, we scale the sine function accordingly:

$$y \approx 3 \cdot \text{sin}\left(\frac{2\pi x}{12}\right)$$

This provides a reasonable first approximation of the seasonal component. As with trend modeling, the parameters can later be refined using optimization techniques to better fit the observed data.

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(x_axis, 3 * np.sin(2 * np.pi * x_axis / 12 ), 'r--', label='scaled sine')
plt.plot(x_axis, detrended, label="data", alpha=0.5)
plt.legend();

### Modeling the Data

As discussed earlier, time series data can generally be thought of as a combination of three components:

1. **Trend component:** the long-term increase or decrease in the data
2. **Seasonal component:** the repeating periodic fluctuations
3. **Residuals (errors):** the remaining stochastic or random fluctuations

Here, we will focus on approximating the data using only the trend and seasonal components, ignoring the residual noise.

By combining the power law trend we fitted earlier with the sine-based seasonal signal, we can construct a model that captures both the long-term increase in $\mbox{CO}_2$ levels and the yearly oscillations. The resulting approximation provides a smooth representation of the underlying patterns in the data, highlighting the main structure while leaving out the unpredictable, random variations.

Mathematically, the model can be expressed as:

$$ y_t \approx f_{\text{trend}}(x_t) + f_{\text{season}}(x_t) $$

where $f_{\text{trend}}(x_t) = c_0 + c \cdot x_t^m$ and $f_{\text{season}}(x_t) = 3\cdot\text{sin}\left(\frac{2\pi x_t}{12}\right)$.

This combination gives us a clear, interpretable representation of the observed $\mbox{CO}_2$ data.

In [ ]:
def fx(x):
    trend = func_powerlaw(x, *popt) 
    seasonality = (3 * np.sin(2 * np.pi * x / 12))
    return trend + seasonality

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(normalized_col['co2_val'], label="original data")
plt.plot(x_axis, fx(x_axis), label="modeled data")
plt.legend();

In [ ]:
plt.figure(figsize=(10, 5))

residuals = normalized_col['co2_val'] - fx(x_axis)
plt.plot(x_axis, residuals);

### Residual Seasonality 

After fitting both the trend and the primary seasonal component, the remaining residuals are relatively small (on average around [−2, 2]), indicating that the model captures much of the structure in the data. However, the fit is not perfect — there is still some residual seasonality present. Although this remaining pattern is less obvious than in the original signal, it becomes more noticeable when we zoom into specific regions (e.g., between indices 60 and 120).

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(range(len(residuals[60: 120])), residuals[60: 120])

plt.axvline(x=12, color='r') 
plt.axvline(x=24, color='r') 
plt.axvline(x=36, color='r') 
plt.axvline(x=48, color='r');

<br>

In these segments, we can still observe a repeating structure occurring approximately every 12 time steps. This indicates that a yearly cycle is still present in the residuals.

The reason for this is that a single sine wave does not fully capture the complexity of the seasonal behavior. Real-world seasonal patterns are often more intricate than a simple sinusoid. To better approximate these patterns, we introduce **higher harmonics**.

A harmonic is a sine wave whose frequency is an integer multiple of a base (fundamental) frequency:

$$ f_n = n \cdot f_1 $$

where `n` is a positive integer. That is, $f_n$ is an integer multiple of the fundamental frequency $f_1$ (first harmonic). Since frequency is inversely related to period, increasing the frequency corresponds to decreasing the period. For example, the second harmonic has twice the frequency (or half the period) of the original signal. 

In [ ]:
plt.figure(figsize=(20, 8))

x = np.arange(0, 60, 0.05)

plt.plot(x, 2 * np.sin(x), 'k-', label='2 sine(x)')
plt.plot(x, 1 * np.sin(2 * x), 'r-', label='1 sine(2*x)')
plt.plot(x, 0.5 * np.sin(4 * x), 'b-', label='0.5 (sine4*x)')
plt.axhline(y=0, linewidth=2, color='g', alpha=.3)
plt.legend(fontsize=16);

<center><img src="https://www.dropbox.com/scl/fi/vqrk9obyqp0uzaxq7ao5u/harmonics.png?rlkey=3quk11grxpwla20yca7x3hppu&st=24r29ziq&dl=1" width="500"/></center>
<center> Figure from Ranade and Xu, AN OVERVIEW OF HARMONICS MODELING AND SIMULATION
</center>

<br>

By incorporating higher harmonics, we can better capture subtle variations in the seasonal pattern. For instance, adding a second harmonic with half the period (6 months instead of 12) refines the model:

```python
seasonality = (3 * np.sin(2 * np.pi * x / 12)) + 
              (0.75 * np.sin(2 * np.pi * x / 6)) 
```

This combination allows the model to represent more complex periodic behavior, improving the overall fit and reducing the residual seasonality still present in the data.

In [ ]:
def fx(x):
    trend = func_powerlaw(x, *popt) 
    seasonality = (3 * np.sin(2 * np.pi * x / 12)) + (0.75 * np.sin(2 * np.pi * x / 6)) 
    return trend + seasonality

residuals = normalized_col['co2_val'] - fx(x_axis)

plt.figure(figsize=(10, 5))
plt.plot(residuals);

In [ ]:
residuals = pd.DataFrame(residuals)
print(residuals.sum())
residuals.head()

### Final Model

We can now construct our final model by combining all the components we have developed:

- A power law trend to capture the long-term increase
- A 12-month seasonal component to model the yearly cycle
- A second harmonic (6-month cycle) to refine and better capture the seasonal fluctuations

By adding these components together, we obtain a model that closely approximates the observed data. The inclusion of the harmonic allows us to capture more subtle variations in the seasonal pattern that a single sine wave could not fully represent.

When we plot the full model against the original data, we can observe that the fit is quite strong. The model successfully captures both the upward trend and the recurring oscillations, while leaving only small residual errors. This demonstrates how combining relatively simple components — trend and multiple seasonal terms — can produce a powerful and interpretable representation of a complex time series.

In [ ]:
def fx(x):
    trend = func_powerlaw(x, *popt) 
    seasonality = (3 * np.sin(2 * np.pi * x / 12)) + (0.75 * np.sin(2 * np.pi * x / 6))
    return trend + seasonality

ax = plt.figure(figsize=(10, 5))

plt.plot(normalized_col['co2_val'], label="original data")
plt.plot(fx(x_axis), linestyle='--', color='r', label="modeled")
plt.legend();

### Predicting Future Data

With our final model in place, we can now use it to forecast future values of $\mbox{CO}_2$. Since our model is defined as a function of time, prediction simply involves extending the time variable beyond the observed data and evaluating the model at those new points.

In this case, we want to predict the next 3 years (36 months). To do this, we extend the time axis to include both the original time points and the additional 36 future steps. We can then plug this extended time axis into our model (trend + seasonality + harmonic) to generate predicted values. Because our model captures both the long-term growth and the repeating seasonal structure, the forecast will continue the upward trend while preserving the cyclical patterns observed in the historical data.

It is important to note, however, that while the model can provide reasonable forecasts, it does not account for unexpected changes or external factors. Additionally, since we ignored the noise component, the predictions will appear smoother than real-world observations, lacking the small random fluctuations present in actual data.

In [ ]:
plt.figure(figsize=(10, 5))

x_axis = np.arange(0, len(normalized_col['co2_val']) + 36)

plt.plot(normalized_col['co2_val'], label="original data")
plt.plot(fx(x_axis), linestyle='--', color='r', label="modeled") 
plt.legend();

### Discovering Autocorrelation in the Residuals

#### How Are the Residuals Correlated?

After modeling and removing the trend and seasonality, it is important to examine whether any structure remains in the residuals. One key question is: *are the residuals truly random, or do they still exhibit dependence over time?*

This is where **autocorrelation** comes in. Autocorrelation measures how strongly a time series is correlated with its past values (lags). If the residuals are purely noise, we would expect little to no correlation across lags. However, if patterns remain, this suggests that our model has not fully captured the underlying structure.

In the code below, we explore autocorrelation by:

- Creating lagged versions of the residuals (e.g., $t−1,t−2,…,t−12$)
- Plotting scatter plots of the original values against their lagged counterparts
- Computing the correlation coefficient for each lag

Each subplot shows how the residual at time $t$ relates to the residual at time $t−lag$. The correlation value displayed in the title quantifies this relationship.

In [ ]:
lags = 12
ncols = 4
nrows = int(np.ceil(lags/ncols))

fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(4*ncols, 4*nrows))
residuals.columns = ['co2_val']

corr_vals = [1.0]
for ax, lag in zip(axes.flat, np.arange(1, lags+1, 1)):
    lag_str = f't-{lag}'
    X = (pd.concat([residuals['co2_val'], residuals['co2_val'].shift(-lag)], axis=1,
                   keys=['y'] + [lag_str]).dropna())

    X.plot(ax=ax, kind='scatter', y='y', x=lag_str)
    corr = X.corr().values[0][1]
    corr_vals.append(corr)
    ax.set_ylabel('Original')
    ax.set_title(f'Lag: {lag_str} (corr={corr:.2f})', fontsize=14)
    ax.set_aspect('equal')
    sns.despine()

fig.tight_layout()

display(X.corr())
X.head()

In [ ]:
plt.figure(figsize=(7, 3))
plt.stem(np.arange(len(corr_vals)), np.abs(corr_vals));

#### Interpreting the Results

The correlation values and the stem plot reveal several important insights:

- Non-zero correlations at multiple lags indicate that the residuals are not purely random.
- Strong correlations at specific lags (e.g., lag 12) suggest a remaining periodic structure.
The particularly high correlation at lag 12 ($\approx$ 0.9) reinforces the presence of a yearly cycle

This confirms that even after modeling with a primary seasonal component and a harmonic, some structure still persists in the data.

In [ ]:
# Using statsmodels 
from statsmodels.tsa.stattools import acf

acf_values = acf(residuals['co2_val'], nlags=36)

fig, ax = plt.subplots(figsize=(9, 4))
ax.stem(range(len(acf_values)), np.abs(acf_values))

ax.set_xticks(range(0, 37, 6))
ax.set_title("Absolute Autocorrelation of Residuals")
ax.set_xlabel("Lag")
ax.set_ylabel("|Correlation|")
ax.grid(True, linestyle='--', alpha=0.5)

plt.show();

In [ ]:
# Using statsmodels 
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(9, 4))
plot_acf(residuals['co2_val'], lags=36, ax=ax)

ax.set_xticks(range(0, 37, 6))
ax.set_title("Autocorrelation of Residuals")
ax.set_xlabel("Lag")
ax.set_ylabel("Correlation")
ax.grid(True, linestyle='--', alpha=0.5)

plt.show();